In [7]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    adjusted_rand_score,
)
import warnings
from src.clust_utils import (
    load_config,
    load_real_datasets,
    prepare_tensor,
    build_model,
    predict_algorithms,
)
from src.clustering.algorithms import ClusteringAlgorithmFactory, ClusteringAlgorithms

warnings.filterwarnings("ignore")


## Helper Functions

We define helper functions to permute data and run the clustering evaluation pipeline.

In [8]:
def permute_dataset(X, labels, mode='none', seed=None):
    """
    Permutes the dataset based on the specified mode.

    Args:
        X (np.ndarray): Feature matrix.
        labels (np.ndarray): Ground truth labels.
        mode (str): 'none', 'row', or 'column'.
        seed (int): Random seed.

    Returns:
        tuple: (X_permuted, labels_permuted)
    """
    rng = np.random.RandomState(seed)
    X_perm = X.copy()
    labels_perm = labels.copy()

    if mode == 'row':
        perm = rng.permutation(X.shape[0])
        X_perm = X[perm, :]
        labels_perm = labels[perm]
    elif mode == 'column':
        perm = rng.permutation(X.shape[1])
        X_perm = X[:, perm]
        # Labels do not change for column permutation

    return X_perm, labels_perm

def run_clustering_evaluation(X, labels_true, selected_algo, factory, seed, dataset_name):
    """
    Runs the clustering algorithm selection and evaluation pipeline.

    Args:
        X (np.ndarray): Feature matrix.
        labels_true (np.ndarray): Ground truth labels.
        selected_algo (str): Name of the algorithm to run.
        factory (ClusteringAlgorithmFactory): Factory to create algorithms.
        seed (int): Random seed.
        dataset_name (str): Name of the dataset (for config loading).

    Returns:
        dict: Dictionary containing ARI scores for CAL and SIL methods.
    """
    algo_n_clusters_param = {
        "kmeans": "n_clusters",
        "kmedians": "n_clusters",
        "spectral_clustering": "n_clusters",
        "ward": "n_clusters",
        "agglomerative": "n_clusters",
        "birch": "n_clusters",
        "gaussian": "n_components",
    }

    # Config DEF sweep
    cal_scores_def, sil_scores_def = [], []

    for n_clusters in range(2, 16):
        # Build DEF config:
        if selected_algo in ["dbscan", "hdbscan", "optics"]:
            continue
        param_name = algo_n_clusters_param[selected_algo]
        algo_config_def = {param_name: n_clusters}

        if selected_algo in [
            "kmeans",
            "kmedians",
            "spectral_clustering",
            "gaussian",
        ]:
            algo_config_def["random_state"] = seed

        if selected_algo == "ward":
            algo_config_def["linkage"] = "ward"

        # === Run DEF config
        algo_instance_def = factory.create_algorithm(selected_algo, algo_config_def)
        try:
            labels_def = algo_instance_def.fit_predict(X)
        except Exception as e:
            # Handle cases where algorithm fails
            labels_def = np.zeros(X.shape[0])

        if len(np.unique(labels_def)) < 2:
            cal_def, sil_def = -1, -1
        else:
            cal_def = calinski_harabasz_score(X, labels_def)
            sil_def = silhouette_score(X, labels_def)

        cal_scores_def.append((n_clusters, cal_def))
        sil_scores_def.append((n_clusters, sil_def))

    # === Best n_clusters
    if selected_algo not in ["dbscan", "hdbscan", "optics"]:
        if cal_scores_def:
            n_clusters_cal_def = max(cal_scores_def, key=lambda x: x[1])[0]
            n_clusters_sil_def = max(sil_scores_def, key=lambda x: x[1])[0]
        else:
             n_clusters_cal_def = n_clusters_sil_def = 2 # Default fallback
    else:
        # Assign dummy values for clustering methods without n_clusters
        n_clusters_cal_def = n_clusters_sil_def = -1

    results = {}

    # === Step 3: 8 combinations of ARI ===
    sources = ["CAL", "SIL"]
    for source in sources:
        if source == "CAL":
            n_clusters_sel = n_clusters_cal_def
        elif source == "SIL":
            n_clusters_sel = n_clusters_sil_def

        # === Build config for this run:
        if Path("configs").exists():
             config_path = Path(f"configs/{dataset_name.lower()}.yaml")
        else:
             config_path = Path("..") / f"configs/{dataset_name.lower()}.yaml"

        if selected_algo in ["dbscan", "hdbscan", "optics"]:
            config_run = load_config(
                file_path=str(config_path),
                variables={"num_clusters": 6, "random_state": seed},
            )  # num_clusters dummy
            algo_config_run = config_run[selected_algo]
            if "n_neighbors" in algo_config_run:
                algo_config_run["connectivity"] = (
                    ClusteringAlgorithms.get_connectivity_graph(
                        X, algo_config_run["n_neighbors"]
                    )
                )
                del algo_config_run["n_neighbors"]
        else:
            config_run = load_config(
                file_path=str(config_path),
                variables={"num_clusters": n_clusters_sel, "random_state": seed},
            )
            algo_config_run = config_run[selected_algo]
            if "n_neighbors" in algo_config_run:
                algo_config_run["connectivity"] = (
                    ClusteringAlgorithms.get_connectivity_graph(
                        X, algo_config_run["n_neighbors"]
                    )
                )
                del algo_config_run["n_neighbors"]
            param_name = algo_n_clusters_param[selected_algo]
            algo_config_run[param_name] = n_clusters_sel
            if selected_algo == "ward":
                algo_config_run["linkage"] = "ward"

        # === Run algorithm:
        algo_instance = factory.create_algorithm(selected_algo, algo_config_run)
        try:
            labels_pred = algo_instance.fit_predict(X)
        except Exception:
            labels_pred = np.zeros(X.shape[0])

        if len(np.unique(labels_pred)) < 2:
            ari_score = 0.0
        else:
            ari_score = adjusted_rand_score(labels_true, labels_pred)

        results[source] = ari_score

    return results

## Experiment Setup

In [9]:
MODEL_NAME = "resnet" # Or whichever model you want to test

# Adjust paths based on CWD
if Path("data/real_world_datasets").exists():
    DATA_PATH = Path("data/real_world_datasets")
    MODEL_PATH = Path(f"models/{MODEL_NAME}.pth")
else:
    DATA_PATH = Path("data/real_world_datasets")
    MODEL_PATH = Path(f"../models/{MODEL_NAME}.pth")

N_PERMUTATIONS = 100
SEED = 42

algorithms = [
    "kmeans",
    "kmedians",
    "spectral_clustering",
    "ward",
    "agglomerative",
    "dbscan",
    "hdbscan",
    "optics",
    "birch",
    "gaussian",
]

factory = ClusteringAlgorithmFactory()

## Load Data and Model

In [10]:
all_data, dataset_names = load_real_datasets(DATA_PATH)
models = build_model(MODEL_NAME, MODEL_PATH)


📥 Loading model from: models\resnet.pth
✅ Model loaded and set to eval mode.


## Run Experiment Function

In [11]:
def run_experiment(permutation_mode='none', n_repeats=1):
    results_list = []

    for i in range(n_repeats):
        current_seed = SEED + i
        print(f"Run {i+1}/{n_repeats} (Mode: {permutation_mode})")

        # 1. Prepare Data (Permute if needed)
        current_data_dict = {}
        for name in dataset_names:
            X = all_data[name]["X_standard"]
            labels = all_data[name]["labels"]

            X_perm, labels_perm = permute_dataset(X, labels, mode=permutation_mode, seed=current_seed)

            current_data_dict[name] = {
                "X_standard": X_perm,
                "labels": labels_perm
            }

        # 2. Prepare Tensor for Model
        # We need to use the helper function but it expects the dictionary structure
        # prepare_tensor uses 'X_standard' key.
        x_padded = prepare_tensor(current_data_dict, dataset_names)

        # 3. Predict Algorithms
        predictions = predict_algorithms(models, x_padded)

        # 4. Evaluate
        for idx, dataset_name in enumerate(dataset_names):
            algo_idx = np.argmax(predictions[idx])
            selected_algo = algorithms[algo_idx]

            X = current_data_dict[dataset_name]["X_standard"]
            labels = current_data_dict[dataset_name]["labels"]

            ari_scores = run_clustering_evaluation(
                X, labels, selected_algo, factory, current_seed, dataset_name
            )

            results_list.append({
                "dataset": dataset_name,
                "permutation_mode": permutation_mode,
                "run_id": i,
                "predicted_algo": selected_algo,
                "ari_cal": ari_scores["CAL"],
                "ari_sil": ari_scores["SIL"]
            })

    return pd.DataFrame(results_list)

## Execute Experiments

In [12]:
# 1. Baseline (No Permutation)
print("Running Baseline...")
df_baseline = run_experiment(permutation_mode='none', n_repeats=1)

# 2. Row Permutation
print("\nRunning Row Permutation...")
df_row = run_experiment(permutation_mode='row', n_repeats=30)

# 3. Column Permutation
print("\nRunning Column Permutation...")
df_col = run_experiment(permutation_mode='column', n_repeats=30)

Running Baseline...
Run 1/1 (Mode: none)

🔮 Predicting best algorithms using trained model...
⏱️ Inference time: 0.1698 seconds
✅ Algorithm prediction complete.

Running Row Permutation...
Run 1/30 (Mode: row)

🔮 Predicting best algorithms using trained model...
⏱️ Inference time: 0.1245 seconds
✅ Algorithm prediction complete.
Run 2/30 (Mode: row)

🔮 Predicting best algorithms using trained model...
⏱️ Inference time: 0.1090 seconds
✅ Algorithm prediction complete.
Run 3/30 (Mode: row)

🔮 Predicting best algorithms using trained model...
⏱️ Inference time: 0.1800 seconds
✅ Algorithm prediction complete.
Run 4/30 (Mode: row)

🔮 Predicting best algorithms using trained model...
⏱️ Inference time: 0.0882 seconds
✅ Algorithm prediction complete.
Run 5/30 (Mode: row)

🔮 Predicting best algorithms using trained model...
⏱️ Inference time: 0.0878 seconds
✅ Algorithm prediction complete.
Run 6/30 (Mode: row)

🔮 Predicting best algorithms using trained model...
⏱️ Inference time: 0.1088 second

## Analysis and Visualization

In [13]:
# Combine Results
df_all = pd.concat([df_baseline, df_row, df_col], ignore_index=True)

# Calculate Stability
# Fraction of permutations where predicted algorithm == baseline prediction
baseline_preds = df_baseline.set_index("dataset")["predicted_algo"].to_dict()

def check_stability(row):
    return row["predicted_algo"] == baseline_preds[row["dataset"]]

df_all["stable_prediction"] = df_all.apply(check_stability, axis=1)

# Group by Dataset and Mode
summary = df_all.groupby(["dataset", "permutation_mode"]).agg(
    mean_ari_cal=("ari_cal", "mean"),
    std_ari_cal=("ari_cal", "std"),
    mean_ari_sil=("ari_sil", "mean"),
    std_ari_sil=("ari_sil", "std"),
    stability=("stable_prediction", "mean")
).reset_index()

print("Summary Statistics:")
display(summary)

Summary Statistics:


,dataset,permutation_mode,mean_ari_cal,std_ari_cal,mean_ari_sil,std_ari_sil,stability
0,BreastTissue,column,0.268214,0.000000,0.268214,0.000000,1.000000
1,BreastTissue,none,0.268214,NaN,0.268214,NaN,1.000000
2,BreastTissue,row,0.268214,0.000000,0.268214,0.000000,1.000000
3,Cervical_Cancer_Risk_Factors,column,0.082761,0.029924,0.103404,0.047637,0.066667
4,Cervical_Cancer_Risk_Factors,none,0.102060,NaN,0.161753,NaN,1.000000
5,Cervical_Cancer_Risk_Factors,row,0.064152,0.044543,0.089005,0.057671,0.000000
6,Ecoli,column,0.522509,0.064997,0.441004,0.165351,0.800000
7,Ecoli,none,0.501089,NaN,0.501089,NaN,1.000000
8,Ecoli,row,0.497933,0.003931,0.497933,0.003931,1.000000
9,Glass,column,0.245294,0.000000,0.245294,0.000000,1.000000


In [15]:
results_dir = Path("results")
results_dir.mkdir(parents=True, exist_ok=True)

output_path = results_dir / "permutation_analysis.csv"
df_all.to_csv(output_path, index=False)

print(f"Saved: {output_path.resolve()}")

Saved: D:\Projects\clustreconet\results\permutation_analysis.csv
